# 01 · Data cleaning and merging

This notebook turns three raw listing datasets into one clean, de-duplicated table:
`data/processed/listings_clean.csv`.

| Source | Kaggle dataset | Raw rows | Location format |
|---|---|---:|---|
| **OLX** | [mohammedaltet/egypt-houses-price](https://www.kaggle.com/datasets/mohammedaltet/egypt-houses-price) | 27,361 | `City` = "New Cairo - El Tagamoa" plus a separate `Compound` column |
| **Bayut** | [ameribrahim/real-estate-bayot-egypt](https://www.kaggle.com/datasets/ameribrahim/real-estate-bayot-egypt) | 50,000 | "Badya Palm Hills Compound, 6th of October, Giza" |
| **Cairo** | [iyadelwy/egypt-housing-prices](https://www.kaggle.com/datasets/iyadelwy/egypt-housing-prices) | 11,418 | "Mivida, 5th Settlement Compounds, The 5th Settlement, New Cairo City, Cairo" |

All the cleaning logic lives in `src/egypt_real_estate/` (`cleaning.py`, `locations.py`), so it is
tested (`tests/`) and reused by every notebook. This notebook runs it and shows what each rule did.

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().resolve().parent / "src"))

import pandas as pd

from egypt_real_estate import config
from egypt_real_estate.cleaning import build_dataset
from egypt_real_estate.locations import match_district

pd.set_option("display.float_format", lambda v: f"{v:,.0f}")

## 1. Run the pipeline

Each step below is applied in order. The table records how many rows every rule removed, so the
cleaning can be audited instead of trusted.

In [2]:
listings, log = build_dataset()
cleaning_log = log.to_frame()
cleaning_log

,source,step,rows_before,rows_after,removed
0,OLX,drop rows with a phone-number placeholder as location,27361,27300,61
1,Bayut,drop repeated listing URLs,50000,45812,4188
2,Bayut,keep EGP prices only (drop USD/other currencies),45812,45812,0
3,Cairo,drop 'Ask' (no price) listings,11418,11006,412
4,All,combine the three sources,84118,84118,0
5,All,drop non-residential / unknown property types,84118,83922,196
6,All,"drop rows missing price, area, bedrooms or bathrooms",83922,82536,1386
7,All,area between 25 and 2000 m²,82536,82449,87
8,All,1-10 bedrooms and bathrooms,82449,82387,62
9,All,"price per m² between 3,000 and 400,000 EGP (removes rentals, typos and unconverted foreign-currency prices)",82387,79272,3115


In [3]:
raw_total = 27_361 + 50_000 + 11_418
print(f"Raw listings:   {raw_total:,}")
print(f"Clean listings: {len(listings):,}  ({len(listings) / raw_total:.0%} kept)")

Raw listings:   88,779
Clean listings: 62,340  (70% kept)


**What the rules fix**

* **Currency.** Bayut has a `currency` column. Only EGP rows are kept, so a USD price can never be
  mixed in with EGP prices. In this snapshot every Bayut row happens to be EGP, so the rule removes 0
  rows, but the check is there. OLX and Cairo have no currency column, so the price-per-m² limits act
  as a second guard: a USD price, a monthly rent or a down payment entered as the price all show up
  as an impossibly low price per m².
* **Property types.** Labels like "Stand Alone Villa", "Standalone Villa" and "iVilla" become
  **Villa**, and "Town House" becomes **Townhouse**. Rooms, land plots, roofs and whole compounds are
  dropped because they aren't comparable homes.
* **Outliers.** The old version cut 1.5 × IQR on *raw price* across the whole market, which removed
  real luxury villas and kept cheap typos. Here the fence is on **log(price per m²)**, computed
  separately for each source and property type.
* **Duplicates.** See section 3.

## 2. Location standardization

The same area is written in many ways: "New Cairo - El Tagamoa" (OLX),
"Cairo Festival City, New Cairo, Cairo" (Bayut), "mivida, 5th settlement compounds, ..., new cairo city, cairo"
(Cairo). Treated as raw text, one area becomes dozens of categories.

`locations.py` maps every raw string to:

* **district**, a canonical area such as *New Cairo*, *Sheikh Zayed* or *North Coast*, using ordered
  pattern rules (specific places first, broad fallbacks last)
* **region**, e.g. Greater Cairo, North Coast, Alexandria, Red Sea & Sinai
* **compound**, the development (e.g. *Mivida*) when the source gives one, normalized so the same
  compound matches across sources

The table shows how many distinct raw spellings collapsed into each of the largest districts.

In [4]:
spellings = (
    listings.groupby("district")
    .agg(listings=("location_raw", "size"),
         raw_spellings=("location_raw", "nunique"),
         example=("location_raw", lambda s: " | ".join(s.drop_duplicates().head(3))))
    .sort_values("listings", ascending=False)
)
pd.set_option("display.max_colwidth", 110)
spellings.head(12)

,listings,raw_spellings,example
district,,,
New Cairo,21275,543,"New Cairo - El Tagamoa | Katameya | 5th Settlement, New Cairo, Cairo"
North Coast,6624,145,North Coast | Alamein | Marsa Matrouh
Sheikh Zayed,5918,127,"Sheikh Zayed | Hills of one Compound, Sheikh Zayed, Giza | El Khamayel Compound, Sheikh Zayed, Giza"
6th of October,5637,114,6th of October | Hadayek 6th of October | West Somid
Mostakbal City,3725,47,"Mostakbal City | Mostakbal City, Cairo | Sarai, Mostakbal City, Cairo"
Ain Sokhna,2754,56,"Ain Sukhna | Ain Sukhna, Suez | Mountain View Sokhna, Ain Sukhna, Suez"
Madinaty,2726,3,"Madinaty | Madinaty, Cairo | Privado, Madinaty, Cairo"
New Capital,2630,100,"New Capital City | Al Maqsad, New Capital City, Cairo | IL Bosco, New Capital City, Cairo"
Shorouk City,2301,81,"Shorouk City | Al Burouj, Shorouk City, Cairo | EL Patio Prime, Shorouk City, Cairo"


In [5]:
print(f"{listings['location_raw'].nunique():,} raw location strings -> "
      f"{listings['district'].nunique()} districts in {listings['region'].nunique()} regions")
print(f"Listings mapped to a named district: {(listings['district'] != 'Other').mean():.1%}")

coverage = (
    listings.assign(named=listings["district"].ne("Other"),
                    has_compound=listings["compound"].notna())
    .groupby("source")[["named", "has_compound"]].mean()
    .rename(columns={"named": "district identified (%)", "has_compound": "compound identified (%)"})
)
(coverage * 100).round(1)

1,884 raw location strings -> 35 districts in 7 regions
Listings mapped to a named district: 99.7%


,district identified (%),compound identified (%)
source,,
Bayut,100,79
Cairo,100,84
OLX,99,55


## 3. Duplicate listings

Brokers often post the same unit several times, and some units appear on more than one site. In the
raw Bayut file, 9,435 rows are exact copies of another row apart from the URL.

Two listings count as the **same unit** when they share district, property type, bedrooms,
bathrooms, area (to the m²) and price (to the nearest EGP 10,000). When duplicates exist, the copy
with a known compound is kept.

In [6]:
dedup = cleaning_log[cleaning_log["step"].str.startswith("remove duplicate")].iloc[0]
print(dedup["step"])
print(f"Removed {dedup['removed']:,} of {dedup['rows_before']:,} rows "
      f"({dedup['removed'] / dedup['rows_before']:.0%})")

remove duplicate listings (3,161 rows were in a group spanning 2+ sites)
Removed 16,699 of 79,039 rows (21%)


## 4. Final dataset

In [7]:
pd.crosstab(listings["type"], listings["source"], margins=True, margins_name="Total")

source,Bayut,Cairo,OLX,Total
type,,,,
Apartment,19503,4515,5918,29936
Chalet,3226,12,3674,6912
Duplex,1567,476,632,2675
Penthouse,1018,385,426,1829
Townhouse,2521,691,2098,5310
Twin House,1365,487,1790,3642
Villa,5858,2505,3673,12036
Total,35058,9071,18211,62340


In [8]:
listings.describe().T[["count", "mean", "50%", "min", "max"]].rename(columns={"50%": "median"})

,count,mean,median,min,max
price,"62,340","9,887,434","6,747,250","130,000","325,000,000"
area_sqm,"62,340",229,182,25,"2,000"
bedrooms,"62,340",3,3,1,10
bathrooms,"62,340",3,3,1,10
price_per_sqm,"62,340","45,421","33,478","3,000","400,000"


In [9]:
config.PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
listings.to_csv(config.PROCESSED_FILE, index=False)
cleaning_log.to_csv(config.REPORTS_DIR / "cleaning_log.csv", index=False)
print(f"Saved {len(listings):,} rows to {config.PROCESSED_FILE.relative_to(config.PROJECT_ROOT)}")

Saved 62,340 rows to data/processed/listings_clean.csv


## Summary

* **88,779 raw rows → 62,340 clean, unique listings** (70% kept). Every removal is logged in
  `reports/cleaning_log.csv`.
* **1,884 raw location strings → 35 districts in 7 regions.** 99.7% of listings map to a named
  district, and 55–84% (depending on source) also have a compound.
* **16,699 duplicate rows removed.** Most are the same unit reposted on one site, and 984
  duplicate groups span two sites.
* The Bayut currency check is in place. All rows in this snapshot are EGP, and the price-per-m²
  limits guard the two sources that have no currency column.

Next: `02_eda.ipynb`.